# Spotify Audio Features vs Popularity — Data Analysis

## Goal
Analyze which audio features are associated with track popularity using Pandas and NumPy.

## Sections
A. Setup
B. Data Cleaning
C. Feature Overview
D. Outlier Handling (IQR)
E. Correlation Analysis
F. Bivariate Analysis
G. Group Comparisons
H. Hit Song Profile
I. Baseline Model (Bonus)

## A. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add src to path to import utils
sys.path.append(os.path.abspath(os.path.join('../src')))
from utils import standardize_columns, detect_and_rename_columns, basic_summary, iqr_filter, savefig

# Configuration
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
np.random.seed(42)

%matplotlib inline

ModuleNotFoundError: No module named 'utils'

In [ ]:
DATA_PATH = '../data/spotify_tracks.csv'

if not os.path.exists(DATA_PATH):
    print(f"WARNING: Data file not found at {DATA_PATH}. Please place the CSV file in the data directory.")
else:
    df = pd.read_csv(DATA_PATH)
    basic_summary(df)

## B. Data Cleaning

In [ ]:
# Standardize column names
df = standardize_columns(df)
df, mapping = detect_and_rename_columns(df)

# Check for essential columns
essential_cols = ['popularity', 'danceability', 'energy', 'loudness', 'tempo', 'valence', 'explicit', 'mode']
missing_cols = [c for c in essential_cols if c not in df.columns]
if missing_cols:
    print(f"Critical Warning: Missing columns: {missing_cols}")
else:
    print("All essential columns present.")

In [ ]:
# Convert types if needed
if 'explicit' in df.columns and df['explicit'].dtype == 'object':
    # Attempt to convert 'True'/'False' strings to bool
    df['explicit'] = df['explicit'].map({'True': True, 'False': False, '1': True, '0': False, 1: True, 0: False})
    df['explicit'] = df['explicit'].astype(bool)
    print("Converted 'explicit' column to boolean.")

# Handle missing values (simple drop as per instructions)
print(f"Shape before dropping NaN: {df.shape}")
df.dropna(inplace=True)
print(f"Shape after dropping NaN: {df.shape}")

# Remove duplicates
initial_len = len(df)
if 'track_id' in df.columns:
    df.drop_duplicates(subset=['track_id'], inplace=True)
    print(f"Dropped duplicates by track_id. Removed: {initial_len - len(df)}")
else:
    # Fallback deduplication
    subset_cols = [c for c in ['track_name', 'artists', 'duration_ms'] if c in df.columns]
    if subset_cols:
        df.drop_duplicates(subset=subset_cols, inplace=True)
        print(f"Dropped duplicates by {subset_cols}. Removed: {initial_len - len(df)}")

df_clean = df.copy()
print(f"Final Clean DataFrame Shape: {df_clean.shape}")

## C. Feature Overview

In [ ]:
# Descriptive statistics
df_clean[essential_cols].describe()

In [ ]:
# Popularity Distribution
plt.figure(figsize=(10, 6))
sns.histplot(df_clean['popularity'], bins=30, kde=True, color='skyblue')
plt.title('Distribution of Track Popularity')
plt.xlabel('Popularity (0-100)')
plt.ylabel('Count')
savefig('popularity_distribution.png')
plt.show()

In [ ]:
# Audio Features Distributions
features_to_plot = ['danceability', 'energy', 'valence', 'acousticness', 'instrumentalness', 'speechiness']
available_features = [f for f in features_to_plot if f in df_clean.columns]

plt.figure(figsize=(15, 10))
for i, col in enumerate(available_features, 1):
    plt.subplot(2, 3, i)
    sns.histplot(df_clean[col], bins=30, kde=True, color='teal')
    plt.title(f'{col.capitalize()} Distribution')
plt.tight_layout()
savefig('audio_features_distribution.png')
plt.show()

## D. Outlier Handling (IQR)

In [ ]:
# IQR for Tempo
if 'tempo' in df_clean.columns:
    df_filtered, removed_tempo = iqr_filter(df_clean, 'tempo')
else:
    df_filtered = df_clean

# IQR for Loudness
if 'loudness' in df_filtered.columns:
    df_filtered, removed_loudness = iqr_filter(df_filtered, 'loudness')

print(f"\nOriginal Shape: {df_clean.shape}")
print(f"Filtered Shape: {df_filtered.shape}")
print(f"Total Rows Removed: {len(df_clean) - len(df_filtered)}")

In [ ]:
# Compare Dist plots (Raw vs Filtered) for Tempo
if 'tempo' in df_clean.columns:
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.boxplot(x=df_clean['tempo'], color='lightcoral')
    plt.title('Tempo (Raw)')
    
    plt.subplot(1, 2, 2)
    sns.boxplot(x=df_filtered['tempo'], color='lightgreen')
    plt.title('Tempo (Filtered)')
    
    savefig('tempo_outliers_comparison.png')
    plt.show()

## E. Correlation Analysis

In [ ]:
numeric_df = df_filtered.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
savefig('correlation_heatmap.png')
plt.show()

In [ ]:
print("Top Positive Correlations with Popularity:")
print(corr_matrix['popularity'].sort_values(ascending=False).head(5))

print("\nTop Negative Correlations with Popularity:")
print(corr_matrix['popularity'].sort_values(ascending=True).head(5))

## F. Bivariate Analysis

In [ ]:
features_vs_pop = ['danceability', 'energy', 'loudness', 'tempo', 'valence']

plt.figure(figsize=(15, 10))

for i, col in enumerate(features_vs_pop, 1):
    if col not in df_filtered.columns:
        continue
    plt.subplot(2, 3, i)
    # Scatter plot with alpha
    plt.scatter(df_filtered[col], df_filtered['popularity'], alpha=0.1, s=10, color='purple')
    
    # Trendline using numpy polyfit
    x = df_filtered[col]
    y = df_filtered['popularity']
    if len(x) > 0:
        m, b = np.polyfit(x, y, 1)
        plt.plot(x, m*x + b, color='red', linewidth=1)
        
    plt.title(f'{col.capitalize()} vs Popularity')
    plt.xlabel(col)
    plt.ylabel('Popularity')

plt.tight_layout()
savefig('bivariate_scatter_plots.png')
plt.show()

## G. Group Comparisons

In [ ]:
# Explicit vs Non-Explicit
if 'explicit' in df_filtered.columns:
    print("Explicit vs Non-Explicit Popularity:")
    print(df_filtered.groupby('explicit')['popularity'].agg(['mean', 'median', 'count']))
    
    plt.figure(figsize=(6, 5))
    sns.boxplot(x='explicit', y='popularity', data=df_filtered, palette='Set2')
    plt.title('Popularity: Explicit vs Non-Explicit')
    savefig('explicit_vs_popularity.png')
    plt.show()

In [ ]:
# Mode (Major vs Minor)
if 'mode' in df_filtered.columns:
    print("\nMode (0=Minor, 1=Major) Popularity:")
    print(df_filtered.groupby('mode')['popularity'].agg(['mean', 'median', 'count']))
    
    plt.figure(figsize=(6, 5))
    sns.boxplot(x='mode', y='popularity', data=df_filtered, palette='Pastel1')
    plt.title('Popularity over Mode (0=Minor, 1=Major)')
    savefig('mode_vs_popularity.png')
    plt.show()

In [ ]:
# Time Trend (if release_date exists)
if 'release_date' in df_filtered.columns:
    # Try to extract year, handle various formats
    # ISO format usually starts with YYYY
    df_filtered['year'] = pd.to_datetime(df_filtered['release_date'], errors='coerce').dt.year
    
    # Check if year extraction was successful (not all NaT)
    if df_filtered['year'].notnull().sum() > 0:
        yearly_pop = df_filtered.groupby('year')['popularity'].mean()
        
        plt.figure(figsize=(12, 6))
        yearly_pop.plot(color='darkblue', marker='o')
        # Rolling mean for smoothing
        yearly_pop.rolling(window=5).mean().plot(color='orange', label='5-Year Rolling Avg')
        
        plt.title('Average Popularity Trend by Year')
        plt.ylabel('Average Popularity')
        plt.legend()
        savefig('popularity_trend_by_year.png')
        plt.show()
    else:
        print("Could not extract valid years from release_date")

## H. Hit Song Profile

In [ ]:
HIT_THRESHOLD = 80
df_filtered['is_hit'] = df_filtered['popularity'] >= HIT_THRESHOLD

profile_cols = ['danceability', 'energy', 'valence', 'acousticness', 'instrumentalness', 'loudness', 'tempo']
hit_profile = df_filtered.groupby('is_hit')[profile_cols].mean()

print("Hit vs Non-Hit Profile (Mean Values):")
print(hit_profile)

# Visualize Difference
# Normalize for visualization (Min-Max scaling for this quick plot, or just raw if ranges similar)
# Audio features are mostly 0-1, except loudness and tempo. 
# Let's visualize only 0-1 features for clarity
cols_0_1 = ['danceability', 'energy', 'valence', 'acousticness', 'instrumentalness']

profile_0_1 = df_filtered.groupby('is_hit')[cols_0_1].mean().T
profile_0_1.plot(kind='bar', figsize=(10, 6))
plt.title('Hit vs Non-Hit: Audio Features (0-1 Scale)')
plt.ylabel('Mean Value')
plt.xticks(rotation=45)
savefig('hit_vs_nonhit_profile.png')
plt.show()

### Insights
1. **Danceability**: Check if hits are more danceable.
2. **Energy**: Do hits have higher energy?
3. **Instrumentalness**: Hits often have very low instrumentalness (vocals are key).
4. **Acousticness**: Modern hits tend to be less acoustic, though this varies.
5. **Explicit**: Check relation in previous section; sometimes explicit content correlates with higher stream counts in certain genres.
6. **Loudness**: Hits are predominantly louder (closer to 0 dB).

## I. (Optional) Simple Baseline Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

# Features to use
feature_cols = ['danceability', 'energy', 'loudness', 'tempo', 'valence', 'acousticness', 'instrumentalness', 'speechiness']
if 'explicit' in df_filtered.columns:
    feature_cols.append('explicit')
if 'mode' in df_filtered.columns:
    feature_cols.append('mode')

# Drop rows where any of these are missing
ml_df = df_filtered.dropna(subset=feature_cols + ['popularity'])

X = ml_df[feature_cols]
y = ml_df['popularity']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train
model = LinearRegression()
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("--- Baseline Linear Regression Model ---")
print(f"R^2 Score: {r2:.4f}")
print(f"MAE: {mae:.2f}")

# Feature Importance
importance = pd.DataFrame({'Feature': feature_cols, 'Coefficient': model.coef_})
print(importance.sort_values(by='Coefficient', ascending=False))